In [1]:
import pandas as pd

gwa01 = pd.read_csv("GWA01.20261110T221140.csv")
gwa02 = pd.read_csv("GWA02.20261111T021141.csv")

print("GWA01 shape:", gwa01.shape)
print("GWA02 shape:", gwa02.shape)

display(gwa01.head())
display(gwa02.head())

GWA01 shape: (29106, 7)
GWA02 shape: (11178, 7)


,Statistic Label,Year,NACE Rev. 2 Activity,Hazardousness,Waste Category,UNIT,VALUE
0,Waste generated,2004,"Agriculture, forestry and fishing (A)",Hazardous and non-hazardous - Total[HAZ_NHAZ],Total waste,Tonnes,8706.0
1,Waste generated,2004,"Agriculture, forestry and fishing (A)",Hazardous and non-hazardous - Total[HAZ_NHAZ],Primary waste (TOTAL minus SEC)[PRIM],Tonnes,NaN
2,Waste generated,2004,"Agriculture, forestry and fishing (A)",Hazardous and non-hazardous - Total[HAZ_NHAZ],Secondary waste (W033+W103+W128_13)[SEC],Tonnes,NaN
3,Waste generated,2004,"Agriculture, forestry and fishing (A)",Hazardous and non-hazardous - Total[HAZ_NHAZ],Chemical and medical wastes (subtotal)[W01-05],Tonnes,0.0
4,Waste generated,2004,"Agriculture, forestry and fishing (A)",Hazardous and non-hazardous - Total[HAZ_NHAZ],Spent solvents[W011],Tonnes,0.0


,Statistic Label,Year,Hazardousness,Waste Category,Waste Management Operation,UNIT,VALUE
0,Waste treated,2004,Hazardous and non-hazardous - Total[HAZ_NHAZ],Total waste,All Waste treatment[TRT],Tonnes,18227850.0
1,Waste treated,2004,Hazardous and non-hazardous - Total[HAZ_NHAZ],Total waste,"Disposal - landfill and other (D1-D7, D12)[DSP...",Tonnes,6404041.0
2,Waste treated,2004,Hazardous and non-hazardous - Total[HAZ_NHAZ],Total waste,"Disposal - landfill (D1, D5, D12)[DSP_L]",Tonnes,6404041.0
3,Waste treated,2004,Hazardous and non-hazardous - Total[HAZ_NHAZ],Total waste,Disposal - incineration (D10)[DSP_I],Tonnes,37413.0
4,Waste treated,2004,Hazardous and non-hazardous - Total[HAZ_NHAZ],Total waste,"Disposal - other (D2-D4, D6-D7)[DSP_OTH]",Tonnes,0.0


In [2]:
hazard_labels = {
    "Hazardous and non-hazardous - Total[HAZ_NHAZ]": "Total",
    "Hazardous[HAZ]": "Hazardous",
    "Non-hazardous[NHAZ]": "Non-hazardous"
}

df1 = gwa01[
    (gwa01["Waste Category"] == "Total waste") &
    (gwa01["UNIT"] == "Tonnes")
].copy()

df1["Hazardousness"] = df1["Hazardousness"].map(hazard_labels)
df1["VALUE"] = pd.to_numeric(df1["VALUE"], errors="coerce")

wide01 = df1.pivot(
    index=["Year", "NACE Rev. 2 Activity"],
    columns="Hazardousness",
    values="VALUE"
).reset_index()

wide01 = wide01.rename(columns={
    "NACE Rev. 2 Activity": "activity",
    "Total": "total_tonnes",
    "Hazardous": "hazardous_tonnes",
    "Non-hazardous": "non_hazardous_tonnes"
})

wide01 = wide01[
    ["Year", "activity", "total_tonnes",
     "hazardous_tonnes", "non_hazardous_tonnes"]
]

wide01["hazardous_share_pct"] = (
    wide01["hazardous_tonnes"]
    / wide01["total_tonnes"].replace(0, float("nan"))
    * 100
)

wide01 = wide01.sort_values(["Year", "activity"])

wide01.to_csv(
    "GWA01_total_waste_by_activity.csv",
    index=False
)

display(wide01.head())
print("Output shape:", wide01.shape)

Hazardousness,Year,activity,total_tonnes,hazardous_tonnes,non_hazardous_tonnes,hazardous_share_pct
0,2004,"Agriculture, forestry and fishing (A)",8706.0,0.0,8706.0,0.000000
1,2004,All NACE activities plus households (TOTAL_HH),24499142.0,724345.0,23774797.0,2.956614
2,2004,Construction (F),11286882.0,0.0,11286882.0,0.000000
3,2004,"Electricity, gas, steam and air conditioning s...",286494.0,1726.0,284768.0,0.602456
4,2004,Households (EP_HH),1702345.0,149.0,1702196.0,0.008753


Output shape: (198, 6)


In [6]:
operation_codes = {
    "TRT": "total_treated_tonnes",
    "DSP_L_OTH": "disposal_landfill_other_tonnes",
    "DSP_L": "landfill_tonnes",
    "DSP_I": "incineration_tonnes",
    "DSP_OTH": "other_disposal_tonnes",
    "RCV_E": "energy_recovery_tonnes",
    "RCV_R_B": "recycling_backfilling_tonnes",
    "RCV_R": "recycling_tonnes",
    "RCV_B": "backfilling_tonnes"
}

df2 = gwa02[
    (gwa02["Waste Category"] == "Total waste") &
    (gwa02["UNIT"] == "Tonnes")
].copy()

df2["Hazardousness"] = df2["Hazardousness"].map(hazard_labels)
df2["VALUE"] = pd.to_numeric(df2["VALUE"], errors="coerce")

df2["operation_code"] = (
    df2["Waste Management Operation"]
    .str.extract(r"\[([^\]]+)\]$")[0]
)

df2["operation_column"] = (
    df2["operation_code"].map(operation_codes)
)

wide02 = df2.pivot(
    index=["Year", "Hazardousness"],
    columns="operation_column",
    values="VALUE"
).reset_index()

wide02 = wide02.reindex(columns=[
    "Year",
    "Hazardousness",
    *operation_codes.values()
])

wide02 = wide02.sort_values(
    ["Year", "Hazardousness"]
)

wide02.to_csv(
    "GWA02_treatment_by_year_hazardousness.csv",
    index=False
)

display(wide02.head())
print("Output shape:", wide02.shape)

operation_column,Year,Hazardousness,total_treated_tonnes,disposal_landfill_other_tonnes,landfill_tonnes,incineration_tonnes,other_disposal_tonnes,energy_recovery_tonnes,recycling_backfilling_tonnes,recycling_tonnes,backfilling_tonnes
0,2004,Hazardous,122734.0,13675.0,13675.0,37303.0,0.0,6025.0,65731.0,NaN,NaN
1,2004,Non-hazardous,18105116.0,6390366.0,6390366.0,110.0,0.0,162528.0,11552112.0,NaN,NaN
2,2004,Total,18227850.0,6404041.0,6404041.0,37413.0,0.0,168553.0,11617843.0,NaN,NaN
3,2006,Hazardous,190951.0,13790.0,13766.0,35121.0,24.0,9919.0,132121.0,NaN,NaN
4,2006,Non-hazardous,22539546.0,7077425.0,7060282.0,95.0,17143.0,131794.0,15330232.0,NaN,NaN


Output shape: (27, 11)
